In [1]:
# Jupyter Notebook: Notebook_1_Data_Annotation.ipynb
# ==============================================================================
# --- 1. УСТАНОВКА И ИМПОРТЫ ---
# ==============================================================================
# !pip install torch torchvision numpy trimesh[easy] pandas scikit-learn plotly pyvista opencv-python tqdm
# Важно! PyVista может потребовать доп. настройки для работы в Jupyter (Jupyter server extension).
# Если 3D-рендеры не работают, это не критично, кластеризация всё равно будет выполнена.

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image
import cv2 as cv
import pyvista as pv
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize
import plotly.express as px
import trimesh
import re
import timm
import torchvision.transforms as T

# Импортируем архитектуры и загрузчики из вашего кода
# (Предполагается, что они находятся в файле utils.py или определены в ячейке выше)
print("Все библиотеки успешно импортированы.")

Все библиотеки успешно импортированы.


In [2]:
# --- ИСПРАВЛЕННАЯ И УЛУЧШЕННАЯ ФУНКЦИЯ ЗАГРУЗКИ STL ---
# Вставьте эту функцию в ячейку, где она была определена, заменив старую.

def load_stl_xyz_only(stl_path, num_points=1024, verbose=False):
    """
    (ИСПРАВЛЕНО) Загружает, сэмплирует, центрирует и нормализует STL с подробным выводом ошибок.
    Установите verbose=True для одного файла, чтобы увидеть детальную отладку.
    """
    try:
        # 1. Загрузка сетки
        mesh = trimesh.load(stl_path, process=True, force='mesh')
        if verbose: print(f"[{stl_path}] Шаг 1: Trimesh загрузил объект типа {type(mesh)}")

        # 2. Обработка сцены (если Trimesh загрузил сцену вместо одной сетки)
        if isinstance(mesh, trimesh.Scene):
            if verbose: print(f"[{stl_path}] -> Это сцена, объединяем геометрию...")
            mesh = mesh.dump(concatenate=True)
        
        # 3. Проверка на наличие вершин
        if not hasattr(mesh, 'vertices') or len(mesh.vertices) == 0:
            if verbose: print(f"[{stl_path}] ОШИБКА: Сетка пуста (нет вершин).")
            return None
        
        # 4. Проверка площади поверхности (важно!)
        # Если площадь очень мала или равна нулю, сэмплирование не удастся.
        if mesh.area < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Площадь поверхности сетки почти равна нулю ({mesh.area}). Невозможно сэмплировать.")
            return None

        # 5. Сэмплирование точек
        points, _ = trimesh.sample.sample_surface(mesh, num_points)
        if verbose: print(f"[{stl_path}] Шаг 2: Сэмплировано {len(points)} точек.")
        
        # 6. Проверка количества точек
        if len(points) < 1: # Даже если мы запросили много, должна быть хотя бы одна
             if verbose: print(f"[{stl_path}] ОШИБКА: Не удалось сэмплировать ни одной точки.")
             return None
        
        # 7. (БЕЗ ИЗМЕНЕНИЙ) Выравнивание количества точек до num_points
        if len(points) < num_points:
            indices = np.random.choice(len(points), num_points, replace=True)
        else:
            indices = np.random.choice(len(points), num_points, replace=False)
        points = points[indices]

        # 8. (БЕЗ ИЗМЕНЕНИЙ) Центрирование и нормализация
        centroid = np.mean(points, axis=0)
        points -= centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))

        # 9. Проверка вырожденности (если все точки в одном месте)
        if max_dist < 1e-6:
            if verbose: print(f"[{stl_path}] ОШИБКА: Облако точек вырождено (все точки в одной координате).")
            return None
        
        points /= max_dist
        if verbose: print(f"[{stl_path}] -> Успешно обработано!")
        
        return points.astype(np.float32)
        
    except Exception as e:
        # 10. Отлов всех остальных ошибок
        if verbose:
            import traceback
            print(f"[{stl_path}] КРИТИЧЕСКАЯ ОШИБКА: Произошло необработанное исключение.")
            traceback.print_exc() # Печатаем полный traceback для детальной отладки
        return None

class PairedStlImageDataset(Dataset):
    """(ПЕРЕРАБОТАНО) Датасет для структуры 1 STL -> 25 изображений."""
    def __init__(self, stl_root, image_root, num_points=4096, image_size=224):
        self.num_points = num_points
        self.stl_root = Path(stl_root)
        self.image_root = Path(image_root)
        all_stl_files = sorted([f for f in self.stl_root.rglob('*.stl') if f.is_file()])
        
        self.paired_files = []
        # Паттерн для поиска 4 цифр в имени файла
        four_digit_pattern = re.compile(r'(\d{4})')

        for stl_path in tqdm(all_stl_files, desc="Сопоставление файлов"):
            match = four_digit_pattern.search(stl_path.stem)
            if not match:
                continue
            
            stl_number = match.group(1) # Извлекаем 4-значный номер
            
            # Ищем все изображения, начинающиеся с этого номера
            # (например, '0001_00.png', '0001_01.png', ...)
            image_paths = sorted(self.image_root.glob(f"{stl_number}_*.png"))
            
            for image_path in image_paths:
                self.paired_files.append((stl_path, image_path))
        
        print(f"\nНайдено {len(all_stl_files)} STL файлов.")
        print(f"Создано {len(self.paired_files)} пар (STL, Изображение) для обучения.")
        
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.paired_files)

    def __getitem__(self, idx):
        stl_path, image_path = self.paired_files[idx]
        points = load_stl_xyz_only(stl_path, self.num_points)
        if points is None: return None
        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = self.image_transform(image)
        except Exception: return None
        return torch.from_numpy(points), image_tensor

def paired_collate_fn(batch):
    """(Без изменений) Collate функция для нового датасета."""
    batch = list(filter(lambda x: x is not None, batch))
    if not batch: return None, None
    points, images = zip(*batch)
    return torch.stack(points), torch.stack(images)

# --- 2. АРХИТЕКТУРА МОДЕЛЕЙ ---

# 2.1 STL ЭНКОДЕР (PointNet++ из вашего кода)

# (ИСПРАВЛЕНО) Вспомогательные функции для PointNet++ в читаемом и рабочем виде
def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long, device=device).view(1, 1, N).repeat(B, S, 1)
    sqrdists = torch.sum((xyz.unsqueeze(1) - new_xyz.unsqueeze(2)) ** 2, -1)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint, self.radius, self.nsample, self.group_all = npoint, radius, nsample, group_all
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last_channel = in_channel + 3
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        if not self.group_all:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            group_idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, group_idx)
            grouped_xyz -= new_xyz.unsqueeze(2)
            if points is not None:
                grouped_points = index_points(points, group_idx)
                features = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                features = grouped_xyz
        else:
            new_xyz = torch.zeros(xyz.shape[0], 1, 3, device=xyz.device)
            grouped_xyz = xyz.view(xyz.shape[0], 1, -1, 3)
            if points is not None:
                features = torch.cat([grouped_xyz, points.view(points.shape[0], 1, -1, points.shape[2])], dim=-1)
            else:
                features = grouped_xyz
        
        features = features.permute(0, 3, 2, 1)
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            features = F.relu(bn(conv(features)))
        
        new_points = torch.max(features, 2)[0].permute(0, 2, 1)
        return new_xyz, new_points

class StlEncoder(nn.Module):
    """(ИСПРАВЛЕНО) Ваш класс Encoder, переименован для ясности."""
    def __init__(self, in_features=3, embedding_dim=256):
        super().__init__()
        # in_channel теперь правильно 0, так как у нас нет доп. фичей кроме xyz
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=in_features-3, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=128, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=256, mlp=[256, 512, 1024], group_all=True)
        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc_embedding = nn.Linear(512, embedding_dim)
    
    def forward(self, xyz):
        l1_xyz, l1_points = self.sa1(xyz, points=None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        _, l3_points = self.sa3(l2_xyz, l2_points)
        x = l3_points.view(xyz.shape[0], -1)
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        embedding = self.fc_embedding(x)
        return F.normalize(embedding, dim=1)

In [5]:
# ==============================================================================
# --- 2. КОНФИГУРАЦИЯ (ИЗМЕНЕННАЯ ДЛЯ ЛУЧШЕГО ОБУЧЕНИЯ) ---
# ==============================================================================
# --- ПУТИ (без изменений) ---
METADATA_PATH = Path("metadata_supcon.csv")
STL_ENCODER_PRETRAINED_PATH = Path("cross_modal_checkpoint.pth")
MODEL_SAVE_PATH = Path("supcon_vit_stl_final.pth")

# --- ПАРАМЕТРЫ ОБУЧЕНИЯ (КЛЮЧЕВЫЕ ИЗМЕНЕНИЯ) ---
BATCH_SIZE = 128 # << УВЕЛИЧЕНО! Это самое важное изменение.
NUM_EPOCHS = 100
EMBEDDING_DIM = 256
PROJECTION_DIM = 256
IMAGE_SIZE = 224
NUM_POINTS = 4096

# --- ГИПЕРПАРАМЕТРЫ ---
# Возвращаемся к более консервативным, но стабильным значениям
LR_VIT = 5e-5 # << Уменьшено. Более низкий LR часто помогает избежать коллапса.
LR_STL = 1e-5 # << Уменьшено.

WEIGHT_DECAY = 1e-5 # Стандартное значение

# Температуру можно оставить низкой, это помогает разделению
SUPCON_TEMPERATURE = 0.07

device = torch.device("cuda")
print(f"Используемое устройство: {device}")
print(f"Путь к файлу разметки: {METADATA_PATH}")
print(f"Модель будет сохранена в: {MODEL_SAVE_PATH}")

Используемое устройство: cuda
Путь к файлу разметки: metadata_supcon.csv
Модель будет сохранена в: supcon_vit_stl_final.pth


In [6]:
# ==============================================================================
# --- 3. НОВЫЕ АРХИТЕКТУРЫ С ПРОЕКЦИОННОЙ ГОЛОВОЙ ---
# ==============================================================================

class ViTImageEncoder(nn.Module):
    """Энкодер изображений на основе ViT с проекционной головой."""
    def __init__(self, embedding_dim=256, projection_dim=256, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=pretrained, num_classes=0)
        vit_embed_dim = self.backbone.embed_dim
        
        # Основной энкодер, который мы будем использовать для инференса
        self.embedding_head = nn.Linear(vit_embed_dim, embedding_dim)
        
        # Проекционная голова, используется ТОЛЬКО во время обучения
        self.projection_head = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, projection_dim)
        )

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding_head(features)
        
        # Во время обучения возвращаем выход проекционной головы
        if self.training:
            projection = self.projection_head(embedding)
            return embedding, projection # Возвращаем оба для потенциального анализа
        
        # Во время инференса (eval) возвращаем только чистый эмбеддинг
        return embedding

class StlEncoderWithProjection(nn.Module):
    """Обертка над вашим StlEncoder для добавления проекционной головы."""
    def __init__(self, embedding_dim=256, projection_dim=256):
        super().__init__()
        self.backbone = StlEncoder(embedding_dim=embedding_dim)
        
        # Проекционная голова, используется ТОЛЬКО во время обучения
        self.projection_head = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, projection_dim)
        )
        
    def forward(self, x):
        embedding = self.backbone(x)
        
        if self.training:
            projection = self.projection_head(embedding)
            return embedding, projection
            
        return embedding

In [7]:
# ==============================================================================
# --- 4. ДАТАСЕТ И ЗАГРУЗЧИК ДАННЫХ (ИСПРАВЛЕННАЯ ВЕРСИЯ) ---
# ==============================================================================
class SupConDataset(Dataset):
    # (Класс SupConDataset остается без изменений)
    def __init__(self, metadata_path, image_size=224, num_points=4096):
        self.metadata = pd.read_csv(metadata_path)
        self.num_points = num_points
        self.image_transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
            T.RandomAffine(degrees=10, translate=(0.1, 0.1)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        filepath, concept_id = row['filepath'], row['concept_id']
        if row['type'] == 'stl':
            points = load_stl_xyz_only(filepath, self.num_points)
            if points is None: return None
            return torch.from_numpy(points), -1, concept_id
        else:
            try:
                image = Image.open(filepath).convert("RGB")
                return self.image_transform(image), 1, concept_id
            except Exception:
                return None

# --- ИЗМЕНЕНИЕ ЗДЕСЬ ---
def supcon_collate_fn(batch):
    """(ИСПРАВЛЕНО) Собирает батч и пропускает его, если в нем нет обеих модальностей."""
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        return None, None, None, None, None

    data, types, labels = zip(*batch)
    
    # Сначала фильтруем данные в обычные списки
    stl_items = [d for d, t in zip(data, types) if t == -1]
    image_items = [d for d, t in zip(data, types) if t == 1]
    
    # КЛЮЧЕВАЯ ПРОВЕРКА: если в батче отсутствует хотя бы одна из модальностей,
    # мы не можем посчитать кросс-модальный лосс, поэтому пропускаем такой батч.
    if not stl_items or not image_items:
        return None, None, None, None, None

    # Только если обе модальности присутствуют, мы безопасно вызываем torch.stack
    stl_data = torch.stack(stl_items)
    image_data = torch.stack(image_items)
    
    # Индексы (без изменений)
    stl_indices = [i for i, t in enumerate(types) if t == -1]
    image_indices = [i for i, t in enumerate(types) if t == 1]
    
    return stl_data, image_data, torch.tensor(labels, dtype=torch.long), stl_indices, image_indices

In [ ]:
# ==============================================================================
# --- 5. ЦИКЛ ОБУЧЕНИЯ (С ИСПРАВЛЕНИЕМ ОШИБКИ BATCHNORM) ---
# ==============================================================================

# --- Импорты (без изменений) ---
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

# --- КОД SUPERVISED CONTRASTIVE LOSS (без изменений) ---
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super(SupervisedContrastiveLoss, self).__init__()
        self.temperature = temperature
    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        similarity_matrix = torch.matmul(embeddings, embeddings.T)
        labels = labels.unsqueeze(1)
        mask = torch.eq(labels, labels.T)
        logits_mask = torch.ones_like(mask).scatter_(1, torch.arange(embeddings.shape[0]).view(-1, 1).to(embeddings.device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(similarity_matrix / self.temperature) * logits_mask
        log_prob = similarity_matrix / self.temperature - torch.log(exp_logits.sum(1, keepdim=True))
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-8)
        loss = -mean_log_prob_pos.mean()
        return loss

# --- Инициализация (без изменений) ---
stl_encoder = StlEncoderWithProjection(EMBEDDING_DIM, PROJECTION_DIM).to(device)
image_encoder = ViTImageEncoder(EMBEDDING_DIM, PROJECTION_DIM).to(device)

if STL_ENCODER_PRETRAINED_PATH.exists():
    print("Инициализация весов STL-энкодера из предобученной модели...")
    checkpoint = torch.load(STL_ENCODER_PRETRAINED_PATH, map_location=device)
    stl_encoder.backbone.load_state_dict(checkpoint['stl_encoder_state_dict'])

# --- Лосс и Оптимизатор (без изменений) ---
loss_func = SupervisedContrastiveLoss(temperature=SUPCON_TEMPERATURE).to(device)
print("Используется ВСТРОЕННАЯ реализация лосса: SupervisedContrastiveLoss")

optimizer = torch.optim.AdamW([
    {'params': image_encoder.parameters(), 'lr': LR_VIT},
    {'params': stl_encoder.parameters(), 'lr': LR_STL}
], weight_decay=WEIGHT_DECAY)

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

# --- Данные (ИЗМЕНЕНИЕ ЗДЕСЬ) ---
train_dataset = SupConDataset(METADATA_PATH, IMAGE_SIZE, NUM_POINTS)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=supcon_collate_fn,
    num_workers=0,
    pin_memory=True,
    # --- ИСПРАВЛЕНИЕ: Добавляем drop_last=True ---
    # Это предотвратит появление батчей размером 1
    drop_last=True
)

# --- Цикл обучения (без изменений) ---
print("\n--- ЗАПУСК СОВМЕСТНОГО ОБУЧЕНИЯ С SUPERVISED CONTRASTIVE LOSS ---")
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    stl_encoder.train()
    image_encoder.train()
    
    total_loss_epoch = 0.0
    p_bar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{NUM_EPOCHS}")
    
    for stl_batch, img_batch, labels, stl_indices, img_indices in p_bar:
        if stl_batch is None or img_batch is None or stl_batch.shape[0] < 2 or img_batch.shape[0] < 2:
            continue
            
        stl_batch, img_batch, labels = stl_batch.to(device), img_batch.to(device), labels.to(device)
        optimizer.zero_grad()
        
        _, stl_projections = stl_encoder(stl_batch)
        _, img_projections = image_encoder(img_batch)
        
        all_projections = torch.zeros(len(labels), PROJECTION_DIM).to(device)
        all_projections[stl_indices, :] = stl_projections
        all_projections[img_indices, :] = img_projections
        
        loss = loss_func(all_projections, labels)
        
        loss.backward()
        
        # --- ИЗМЕНЕНИЕ: Добавляем градиентное клипирование ---
        torch.nn.utils.clip_grad_norm_(image_encoder.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(stl_encoder.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss_epoch += loss.item()
        p_bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")

    scheduler.step()
    avg_loss = total_loss_epoch / len(train_loader)
    print(f"Эпоха {epoch+1}/{NUM_EPOCHS} | Средний Loss: {avg_loss:.4f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'stl_encoder_state_dict': stl_encoder.state_dict(),
            'image_encoder_state_dict': image_encoder.state_dict(),
            'epoch': epoch + 1,
            'loss': avg_loss
        }, MODEL_SAVE_PATH)
        print(f"Модель сохранена в {MODEL_SAVE_PATH} (лучший loss)")

print("\nОбучение завершено.")

Инициализация весов STL-энкодера из предобученной модели...
Используется ВСТРОЕННАЯ реализация лосса: SupervisedContrastiveLoss

--- ЗАПУСК СОВМЕСТНОГО ОБУЧЕНИЯ С SUPERVISED CONTRASTIVE LOSS ---


Эпоха 1/100:   0%|          | 0/110 [00:00<?, ?it/s]

Эпоха 1/100 | Средний Loss: 0.9971
Модель сохранена в supcon_vit_stl_final.pth (лучший loss)


Эпоха 2/100:   0%|          | 0/110 [00:00<?, ?it/s]

Эпоха 2/100 | Средний Loss: 0.7518
Модель сохранена в supcon_vit_stl_final.pth (лучший loss)


...

Эпоха 27/100:   0%|          | 0/110 [00:00<?, ?it/s]

Эпоха 27/100 | Средний Loss: 0.3219
Модель сохранена в supcon_vit_stl_final.pth (лучший loss)


Эпоха 28/100:   0%|          | 0/110 [00:00<?, ?it/s]

Эпоха 28/100 | Средний Loss: 0.2938
Модель сохранена в supcon_vit_stl_final.pth (лучший loss)


Эпоха 29/100:   0%|          | 0/110 [00:00<?, ?it/s]